# Amnesic Sweep — Can we scale the +0.9pp positive?

Current state: amnesic_all @ L17 α=1.0 last_prompt → **+0.9–3.0pp Pareto-positive (p=0.002 pooled)**.

Goal: find a configuration that gives larger effect while preserving the zero-loss Pareto property.

**8 configs tested on N=200 items (Stage B 10-opt subset, ~20 min)**:

| # | Config | Hypothesis |
|---|---|---|
| 1 | baseline (no ablation) | control |
| 2 | L17 α=1.0 last_prompt (replicate) | reproducibility |
| 3 | L17 α=1.5 last_prompt | overcorrection: invert past zero |
| 4 | L17 α=2.0 last_prompt | full direction inversion |
| 5 | L17 α=1.0 last 5 prompt tokens | cumulative exposure |
| 6 | L11+L17+L23 α=1.0 simultaneous | multi-layer (Zhang & Nanda) |
| 7 | L11 α=1.0 last_prompt | different layer (higher AUROC) |
| 8 | L17 argmax α=1.5 last_prompt | targeted + stronger |

Any config with **Δ > +3pp AND lost = 0** = scalable positive → paper update.

## Cell 1 — Imports (model should be loaded already)

In [ ]:
import json, re, time, random
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from tqdm.auto import tqdm
from huggingface_hub import snapshot_download
from safetensors import safe_open
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

OUT = Path('/content/amnesic_sweep'); OUT.mkdir(exist_ok=True)
print('imports OK')

## Cell 2 — Fit probes at L11, L17, L23 + build letter directions per layer

In [ ]:
corpus = snapshot_download('caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
                            repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

MIN_LEN = 200; POOL_WINDOW = 10
LAYERS = [11, 17, 23]

pooled_by_layer = {L: [] for L in LAYERS}
labels = []

for shard in tqdm(shards, desc='collect activations'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        offs_all = json.loads(meta['offsets'])
        rollouts = json.loads(meta['rollouts'])
        acts_dict = {L: f.get_tensor(f'acts_L{L}') for L in LAYERS}
        for r_idx, r in enumerate(rollouts):
            if r['pred'] is None or r['response_len'] < MIN_LEN: continue
            for L in LAYERS:
                offs = offs_all[f'L{L}']
                ra = acts_dict[L][offs[r_idx]:offs[r_idx+1]].float().numpy()
                pooled_by_layer[L].append(ra[:POOL_WINDOW].mean(axis=0))
            labels.append(r['pred'])

labels = np.array(labels)
letters = sorted(set(labels))
letter_to_int = {l: i for i, l in enumerate(letters)}
y = np.array([letter_to_int[l] for l in labels])
print(f'n={len(labels)}, letters={letters}')

# Fit per-layer probe + build direction stack
d_stacks = {}
probes_gpu = {}  # for argmax mode

for L in LAYERS:
    X = np.stack(pooled_by_layer[L])
    scaler = StandardScaler().fit(X)
    pca = PCA(n_components=128, random_state=42).fit(scaler.transform(X))
    lr = LogisticRegression(C=1.0, max_iter=3000, random_state=42).fit(
        pca.transform(scaler.transform(X)), y)
    print(f'L{L} train: {lr.score(pca.transform(scaler.transform(X)), y):.3f}')

    dirs = []
    for letter_idx in range(len(letters)):
        coef = lr.coef_[letter_idx]
        d_scaled = pca.components_.T @ coef
        d_raw = d_scaled * scaler.scale_
        d_raw = d_raw / (np.linalg.norm(d_raw) + 1e-8)
        dirs.append(torch.from_numpy(d_raw.astype(np.float32)).cuda().to(torch.bfloat16))
    d_stacks[L] = torch.stack(dirs)  # [10, d_model]
    
    probes_gpu[L] = dict(
        mean=torch.from_numpy(scaler.mean_.astype(np.float32)).cuda(),
        scale=torch.from_numpy(scaler.scale_.astype(np.float32)).cuda(),
        pca_T=torch.from_numpy(pca.components_.astype(np.float32)).cuda(),
        coef=torch.from_numpy(lr.coef_.astype(np.float32)).cuda(),
        intercept=torch.from_numpy(lr.intercept_.astype(np.float32)).cuda())

print(f'\n✅ probes + directions for layers {LAYERS}')

def probe_argmax(x, L):
    p = probes_gpu[L]
    xs = (x.float() - p['mean']) / p['scale'].clamp(min=1e-6)
    xp = p['pca_T'] @ xs
    return (p['coef'] @ xp + p['intercept']).argmax().item()

## Cell 3 — Multi-variant AmnesicHook

In [ ]:
def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers'),('layers',)]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

class AmnesicHook:
    """Multi-layer, multi-position, alpha-scaled amnesic ablation.
    mode: 'off' | 'all' | 'argmax'
    target_positions: list of absolute indices (or None for last)
    alpha: scaling coefficient
    layer_idx: which layer this hook is on (for probe selection in argmax mode)
    """
    def __init__(self, layer_idx):
        self.layer_idx = layer_idx
        self.mode = 'off'
        self.target_positions = None
        self.alpha = 1.0
        self.applied = set()  # per-position applied flag

    def set(self, mode, target_positions, alpha):
        self.mode = mode
        self.target_positions = target_positions
        self.alpha = alpha
        self.applied = set()

    def off(self):
        self.mode = 'off'
        self.applied = set()

    def make_hook(self):
        def hook(module, inp, out):
            if self.mode == 'off': return out
            hidden = out[0] if isinstance(out, tuple) else out
            T = hidden.shape[1]
            # Resolve target positions
            if self.target_positions is None:
                positions = [T - 1]
            else:
                positions = [p if p >= 0 else T + p for p in self.target_positions]
                positions = [p for p in positions if 0 <= p < T]
            if not positions: return out
            # Apply ablation at each position (only those not already applied)
            to_apply = [p for p in positions if p not in self.applied]
            if not to_apply: return out
            hidden = hidden.clone()
            ds = d_stacks[self.layer_idx]  # [10, d]
            for p in to_apply:
                x = hidden[:, p, :]  # [B, d]
                if self.mode == 'all':
                    coefs = x @ ds.T  # [B, 10]
                    delta = coefs @ ds  # [B, d]
                    hidden[:, p, :] = x - self.alpha * delta
                elif self.mode == 'argmax':
                    for b in range(x.shape[0]):
                        idx_arg = probe_argmax(x[b], self.layer_idx)
                        d = ds[idx_arg]
                        c = (x[b] * d).sum()
                        hidden[b, p, :] = x[b] - self.alpha * c * d
                self.applied.add(p)
            if isinstance(out, tuple): return (hidden,) + out[1:]
            return hidden
        return hook

# Install hook on each layer
hooks = {}
handles = {}
for L in LAYERS:
    hooks[L] = AmnesicHook(L)
    module = get_layer_module(model, L)
    handles[L] = module.register_forward_hook(hooks[L].make_hook())
print(f'✅ amnesic hooks installed on {LAYERS}')

## Cell 4 — Load eval set (Stage B 10-opt, ≥1 correct for volume)

In [ ]:
questions = []
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        rollouts_ = json.loads(meta['rollouts'])
        opts = json.loads(meta['options'])
        n_correct = sum(1 for r in rollouts_ if r['correct'])
        if len(opts) == 10 and n_correct >= 1:  # 10-option match + at least 1 correct
            questions.append(dict(
                question=meta['question'],
                options=opts,
                gold_letter=meta['gold_letter'],
                n_options=10,
                n_correct=n_correct))
print(f'{len(questions)} Stage B 10-opt questions')

def format_mcq(question, options):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(options))
    content = ("Answer the following multiple-choice question. "
        "Give ONLY the letter of the correct answer.\n\n"
        f"Question: {question}\n\nOptions:\n{choices}\n\nAnswer: \\boxed{{")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)

letter_ids = {L: tok(L, add_special_tokens=False).input_ids[0] for L in 'ABCDEFGHIJ'}
print('letter_ids:', letter_ids)

## Cell 5 — Sweep: 8 configs × N items (~15-20 min)

In [ ]:
N_EVAL = 200
random.seed(42)
sample = random.sample(questions, min(N_EVAL, len(questions)))
print(f'Evaluating {len(sample)} questions')

# Config list: (name, mode, alpha, target_layers, target_positions_relative_to_prompt_end)
# target_positions_relative: None=last, [0]=last, [-4,-3,-2,-1,0]=last 5 tokens
CONFIGS = [
    ('baseline',         'off',    1.0, [],            None),
    ('L17_a1.0',         'all',    1.0, [17],          None),
    ('L17_a1.5',         'all',    1.5, [17],          None),
    ('L17_a2.0',         'all',    2.0, [17],          None),
    ('L17_a1.0_last5',   'all',    1.0, [17],          [-4, -3, -2, -1, 0]),
    ('multi_a1.0',       'all',    1.0, [11, 17, 23],  None),
    ('L11_a1.0',         'all',    1.0, [11],          None),
    ('L17_argmax_a1.5',  'argmax', 1.5, [17],          None),
]

def set_configs(mode, alpha, target_layers, pos_rel, last_pos):
    for L in LAYERS: hooks[L].off()
    if mode == 'off': return
    if pos_rel is None:
        positions = None  # hook uses last by default
    else:
        positions = [last_pos + p for p in pos_rel]
    for L in target_layers:
        hooks[L].set(mode, positions, alpha)

results = []
t0 = time.time()
for i, q in enumerate(tqdm(sample, desc='amnesic sweep')):
    try:
        p = format_mcq(q['question'], q['options'])
        ids = tok(p, return_tensors='pt').input_ids.to('cuda')
        if ids.shape[1] > 4096: continue
        last_pos = ids.shape[1] - 1

        row = dict(idx=i, gold=q['gold_letter'], n_options=q['n_options'])
        for (name, mode, alpha, target_layers, pos_rel) in CONFIGS:
            set_configs(mode, alpha, target_layers, pos_rel, last_pos)
            with torch.no_grad():
                out = model(input_ids=ids, attention_mask=torch.ones_like(ids))
            logits = out.logits[0, -1]
            letter_logits = {L: logits[letter_ids[L]].item() for L in 'ABCDEFGHIJ'[:q['n_options']]}
            pred = max(letter_logits, key=letter_logits.get)
            row[f'{name}_pred'] = pred
            row[f'{name}_correct'] = (pred == q['gold_letter'])
        for L in LAYERS: hooks[L].off()
        results.append(row)
        if (i+1) % 20 == 0:
            accs = {name: sum(r[f'{name}_correct'] for r in results) / len(results) for (name,*_) in CONFIGS}
            top = sorted(accs.items(), key=lambda x: -x[1])[:4]
            top_str = ' | '.join(f'{n}={a*100:.0f}%' for n,a in top)
            print(f'  [{i+1}/{N_EVAL}] top4: {top_str}')
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue

with open(OUT/'sweep_results.json', 'w') as f:
    json.dump(dict(n=len(results), configs=[c[0] for c in CONFIGS], results=results,
                   wall_min=(time.time()-t0)/60), f, indent=2)
print(f'\n✅ {(time.time()-t0)/60:.1f} min')

## Cell 6 — Analysis: sorted by effect, with McNemar stats

In [ ]:
from scipy.stats import binomtest

print(f'=== Amnesic Sweep (N={len(results)}) ===\n')
stats_rows = []
baseline_correct = [r['baseline_correct'] for r in results]
for (name, *_) in CONFIGS:
    if name == 'baseline': continue
    cfg_correct = [r[f'{name}_correct'] for r in results]
    acc = sum(cfg_correct) / len(results)
    base_acc = sum(baseline_correct) / len(results)
    delta = (acc - base_acc) * 100
    gained = sum(1 for bc, cc in zip(baseline_correct, cfg_correct) if not bc and cc)
    lost = sum(1 for bc, cc in zip(baseline_correct, cfg_correct) if bc and not cc)
    n_dis = gained + lost
    p = binomtest(gained, n_dis, p=0.5, alternative='two-sided').pvalue if n_dis > 0 else 1.0
    stats_rows.append((name, acc * 100, delta, gained, lost, p))

# Sort by Δ descending
stats_rows.sort(key=lambda r: -r[2])
base_acc = sum(baseline_correct) / len(results) * 100
print(f'{"baseline":22s}: {base_acc:.1f}%\n')
print(f'{"config":22s}  {"acc%":>6s}  {"Δpp":>6s}  {"gain":>5s}  {"lost":>5s}  {"p":>7s}')
for name, acc, delta, g, l, p in stats_rows:
    star = ' ⭐' if delta > 3 and l == 0 else (' ✓' if delta > 1 and l == 0 else '')
    print(f'{name:22s}: {acc:5.1f}%  {delta:+5.1f}  {g:>5d}  {l:>5d}  {p:>7.3f}{star}')

# Best config that maintains zero-loss Pareto
best_pareto = max([r for r in stats_rows if r[4] == 0], key=lambda r: r[2], default=None)
best_overall = max(stats_rows, key=lambda r: r[2])
print('\n=== VERDICT ===')
if best_pareto and best_pareto[2] > 3:
    print(f'⭐⭐ SCALED POSITIVE (Pareto): {best_pareto[0]} gives {best_pareto[2]:+.1f}pp, 0 losses')
    print(f'    → evolve current {0.9:.1f}pp to {best_pareto[2]:.1f}pp while preserving zero-loss')
elif best_pareto and best_pareto[2] > 1.5:
    print(f'⭐ Moderate scaled positive (Pareto): {best_pareto[0]} gives {best_pareto[2]:+.1f}pp, 0 losses')
elif best_overall[2] > 3:
    print(f'✓ Larger overall effect: {best_overall[0]} gives {best_overall[2]:+.1f}pp, but {best_overall[4]} losses (not Pareto)')
else:
    print(f'⚖️  No scaling: best is {best_overall[0]} at {best_overall[2]:+.1f}pp')